# 07. Azure Synapse Analytics & Storage Setup

## 1. What is Azure Synapse Analytics?
* Formerly called **Azure SQL Data Warehouse**.
* It is an **end-to-end enterprise analytics solution**, not just a standalone data warehouse.
* Unlike other clouds where you need separate services for storage, orchestration, and processing (e.g., AWS Redshift only stores the Core layer), Synapse combines everything in one workspace:
  * **Orchestration / ETL:** Embedded Azure Data Factory (Pipelines).
  * **Data Processing:** Apache Spark Pools & low-code **Mapping Data Flows**.
  * **Data Warehousing:** Serverless & Dedicated SQL Pools (Staging, Core Layer, and Data Marts).

## 2. Azure Free Account & Portal Basics
* **Free Account:** Search *"Azure free account"* → **Try Azure for free**.
  * Provides **$200 USD credit** for **30 days** (cannot be charged after 30 days; services simply nullify).
  * Includes **12 months** of free access to popular services + **always-free** services.
  * Requires a Microsoft account (Gmail/Outlook registered under Microsoft) and card details for verification only.
* **Portal URL:** `portal.azure.com`
* **Resource Group:** A logical **folder** in Azure that holds all your related services/resources (Data Lake, Synapse Analytics, Data Factory, etc.) in one place.

## 3. Step-by-Step Resource Setup

### Step 1: Create a Resource Group
1. Search **Resource groups** in the Azure Portal → Click **+ Create**.
2. **Resource Group Name:** `RG_Azure_Data_Warehouse` *(or any descriptive name)*.
3. **Region:** `Canada East` *(or any available region)*.
4. Click **Review + create** → **Create** → **Go to resource group**.

### Step 2: Create Azure Synapse Analytics Workspace
1. Inside your Resource Group, click **+ Create** → Search **Azure Synapse** → Select **Azure Synapse Analytics** *(by Microsoft)* → **Create**.
2. **Basics Tab:**
   * **Resource Group:** Select the one you just created.
   * **Managed Resource Group:** Leave blank *(Azure auto-generates it)*.
   * **Workspace Name:** Must be globally unique and **lowercase only** *(e.g., `azuredatawarehousing`)*.
   * **Region:** Pick a popular region to avoid networking/availability issues *(e.g., West Europe, UK South, Canada Central, East Asia)*.
   * **Data Lake Storage Gen2 (Account Name):** Synapse requires a primary/managed Data Lake during setup (create new or pick existing).
     * *Best Practice:* In production projects, do **not** use this managed Data Lake; create and connect an **external Data Lake** for better security and governance.
3. **Security Tab:**
   * Create a **SQL Admin username** *(e.g., `admin`)* and a strong **Password**.
4. **Networking Tab:**
   * Enable **Allow all connections** *(for practice)*.
5. **Review + Create:**
   * *Pricing Note:* Serverless SQL Pool costs **~$5 USD per TB** of data processed.
   * Click **Create** *(deployment takes a few minutes)*.

### Step 3: Create an External Data Lake (ADLS Gen2)
1. Go back to your Resource Group → Click **+ Create** → Search **Storage account** *(by Microsoft)* → **Create**.
2. **Basics Tab:**
   * **Storage Account Name:** Must be globally unique *(e.g., `azuredatawarehouselake`)*.
   * **Region:** Same region as your Synapse Workspace.
   * **Performance:** `Standard`.
   * **Redundancy:** **LRS (Locally Redundant Storage)** — the cheapest option.
3. **Advanced Tab (Crucial!):**
   * Check **Enable hierarchical namespace**.
   * *Why?* Without this, it stays a standard Blob Storage account and will not support Data Lake folder structures.
4. Click **Review + create** → **Create** → **Go to resource**.

### Step 4: Configure Containers & Upload Source Data
Inside a Storage Account, **Data storage** provides 4 services:
* **Containers:** Data Lake storage *(used for analytics pipelines)*.
* **File shares:** Central repository to share files with a team *(similar to SharePoint)*.
* **Queues:** Message storage *(similar to Kafka)*.
* **Tables:** Semi-structured NoSQL storage *(similar to Cosmos DB)*.

**Create 4 Containers** (`Data storage` → `Containers` → `+ Container`):
1. `source` — Acting as the raw data source *(using a Data Lake as a source instead of a SQL DB is common in modern pipelines)*.
2. `bronze` — Staging / Raw layer *(as-is data; can be transient/temporary or persistent/permanent)*.
3. `silver` — Transform / Enriched layer.
4. `gold` — Core / Curated layer.

**Upload Initial Data:**
* Open the `source` container → Click **Upload** → Upload the initial batch file (`day0.csv`) first *(keep `day1.csv` aside for testing Incremental Loading later)*.

## 4. Azure Synapse Studio Tour
Open your deployed Synapse Workspace and launch **Synapse Studio**:

| Tab | Purpose & Key Details |
| :--- | :--- |
| **Home** | Quick links for *Ingest*, *Explore & Analyze*, and *Visualize* (rarely used in practice). |
| **Data** | **Workspace:** Create and manage SQL Warehouses & Lake databases.<br>**Linked:** View and connect external Storage Accounts and datasets. |
| **Develop** | Write and save **SQL Scripts** (to build the Data Warehouse), Spark Notebooks, and **Mapping Data Flows**. |
| **Integrate** | **Embedded Azure Data Factory** (just renamed). Build ETL pipelines, Copy activities, and orchestration workflows. |
| **Monitor** | Monitor pipeline runs, SQL requests, and **Apache Spark Pools** (Spark pools are a key feature available here that ADF lacks). |
| **Manage** | Configure **Linked Services**, SQL Pools, Spark Pools, and security. |

### Key Terms to Remember
* **Linked Service:** A connection configuration between Synapse and an external source or destination (e.g., connecting Synapse to an ADLS Gen2 Data Lake).
* **Mapping Data Flows:** A **low-code / no-code** drag-and-drop transformation tool in Synapse. Behind the scenes, it automatically runs **Apache Spark** code.

## Bronze & Silver Layers (Ingestion, Incremental Loading & Transformations)

## 1. Raw Ingestion (Bronze Layer)
The **Bronze (Staging)** layer stores raw, as-is data extracted from source systems. In Synapse, we use the **Copy data** activity inside a pipeline to move data from the `source` container to `bronze`.

### Prerequisite: Grant Synapse Access to External Data Lake (RBAC)
Because the external ADLS Gen2 Data Lake is a separate resource from the Synapse Workspace, you must authorize Synapse to read/write to it using **System-Assigned Managed Identity**:
1. Open your external **Storage Account** (`azuredatawarehouselake`) in the Azure Portal.
2. Go to **Access Control (IAM)** → Click **+ Add** → **Add role assignment**.
3. **Role:** Search and select **Storage Blob Data Contributor** → Click **Next**.
4. **Members:** Select **Managed identity** → Click **+ Select members**.
5. Choose **Synapse workspace** → Select your workspace → Click **Select**.
6. Click **Review + assign**.

### Step-by-Step: Build the Raw Ingestion Pipeline
1. In Synapse Studio, go to the **Integrate** tab → Click **+** → **Pipeline**.
2. Name the pipeline `Raw_Ingestion`.
3. Search for **Copy data** in the Activities pane and drag it onto the canvas. Rename it to `Raw_Ingestion`.

#### Configure Source (CSV)
1. Go to the **Source** tab → Click **+ New** next to Source dataset.
2. Select **Azure Data Lake Storage Gen2** → Format: **DelimitedText (CSV)**.
3. **Dataset Name:** `DS_Raw_CSV`.
4. **Linked Service:** Click **+ New**:
   * **Name:** `LS_DataLake`
   * **Authentication type:** `System Assigned Managed Identity`
   * **Storage account name:** Select your external Data Lake (`azuredatawarehouselake`).
   * Click **Test connection** (should succeed) → **Create**.
5. **File path:** Browse to `source` → select `day0.csv` → Click **OK**.

#### Configure Sink (Parquet)
> **Best Practice:** Always convert raw CSV files to **Parquet** in the Sink. Unlike CSV, Parquet is columnar, compressed, and **preserves schema metadata**.

1. Go to the **Sink** tab → Click **+ New** next to Sink dataset.
2. Select **Azure Data Lake Storage Gen2** → Format: **Parquet**.
3. **Dataset Name:** `DS_Parquet`.
4. **Linked Service:** Reuse `LS_DataLake`.
5. **File path:**
   * **Container:** `bronze`
   * **Directory:** `sales`
   * **File name:** `order_day0.parquet`
6. **Import schema:** Select **None** *(the destination file does not exist yet)* → Click **OK**.

#### Configure Schema Mapping & Run
1. Go to the **Mapping** tab → Click **Import schemas** to pull column definitions from the source.
2. Adjust target data types as needed:
   * `order_id` → `String`
   * `order_date` → `Date`
   * `unit_price` → `Int64`
   * `total_amount` → `Int64`
3. Click **Debug** to run the pipeline.
4. Verify in ADLS Gen2 that `bronze/sales/order_day0.parquet` is created.
5. Click **Publish all** to save your work.

## 2. Incremental Loading

### What is Incremental Loading?
Instead of reloading the entire historical dataset every time a pipeline runs, **Incremental Loading** processes **only the delta** (new or modified data since the previous load).

* **Example:** If you loaded data from Jan 1 to Apr 13 on Day 0, on Apr 14 you only load the 1 day of new data (Apr 13 → Apr 14), rather than reloading Jan 1 → Apr 14.
* **Benefits:** Reduces compute cost, speeds up pipelines, avoids duplicate data, and minimizes load on source systems.
* *Note:* Implementation depends on the source type (e.g., watermark/timestamp columns for SQL databases, new files/folders for Data Lakes, pagination/timestamps for APIs).

### Step-by-Step: File-Based Incremental Load
1. Upload the new batch file (`day1.csv`) to the `source` container in ADLS Gen2.
2. Open the `Raw_Ingestion` pipeline → Select the Copy Activity:
   * **Source tab:** Open `DS_Raw_CSV` → Update the file path to `day1.csv`.
   * **Sink tab:** Open `DS_Parquet` → Update the file name to `order_day1.parquet`.
3. Click **Debug** to run the pipeline.
4. Verify `bronze/sales/` now contains both `order_day0.parquet` and `order_day1.parquet` (creating a **persistent** staging layer with full history).
5. Click **Publish all**.

## 3. Data Transformation (Silver Layer via Mapping Data Flows)
The **Silver (Enriched / Transform)** layer cleans, filters, and enriches raw data from Bronze.

### What is a Mapping Data Flow?
A **low-code / no-code** visual transformation tool in Synapse. You build transformations using drag-and-drop blocks, and Synapse executes **Apache Spark** code behind the scenes.

### Step-by-Step: Build the Transformation Pipeline
1. Go to **Integrate** → Create a new Pipeline named `Transformation_Layer`.
2. Drag a **Data flow** activity onto the canvas → Rename it to `Data_Transformation`.
3. Go to the **Settings** tab → Click **+ New** next to Data flow → Name it `Data_Flow`.

#### Step 1: Add Source (Bronze)
1. Click **Add Source** on the canvas → Output stream name: `Source_Bronze`.
2. Click **+ New** next to Dataset:
   * Select **ADLS Gen2** → **Parquet** → Name: `DS_Parquet_Bronze`.
   * **Linked Service:** `LS_DataLake`.
   * **File path:** Container `bronze`, Directory `sales`, **leave File name blank** *(reads all Parquet files in the folder)*.
3. **Useful Source Options:**
   * **Wildcard paths:** Used to read files across nested subfolders.
   * **After completion (Delete source files):** Check this only if you want a **transient** Bronze layer; leave unchecked to keep it **persistent**.
4. **Enable Data Flow Debug:**
   * Toggle **Data flow debug** **ON** at the top bar (takes ~2–3 minutes to spin up a Spark cluster; default TTL is 1 hour).
   * *Cost Tip:* Turn Debug **OFF** when finished developing. Production pipeline runs use the AutoResolveIntegrationRuntime and do not require debug mode.
5. Open the **Data preview** tab → Click **Refresh** to verify all records are loaded (e.g., 10 records total across `day0` and `day1`).

#### Step 2: Apply Transformations
1. **Filter Transformation** *(remove orders with unit price < 100)*:
   * Click **+** next to `Source_Bronze` → Select **Filter**.
   * **Output stream name:** `Filter_Less_Than_100`.
   * Open **Expression Builder** → Enter: `unit_price >= 100` → **Save and finish**.
   * Preview data to confirm rows below 100 are excluded.

2. **Derived Column — Split Email** *(equivalent to `withColumn()` in PySpark)*:
   * Click **+** → Select **Derived Column** → Name: `Add_Domain`.
   * **Column:** Type a new column name `domain`.
   * **Expression:** `split(customer_email, '@')` *(splits the email into a 2-element array: `['username', 'example.com']`)*.

3. **Derived Column — Extract Domain from Array**:
   * Click **+** → Select **Derived Column** → Name: `Fetch_Only_Domain`.
   * **Column:** Select the existing `domain` column.
   * **Expression:** `slice(domain, 2, 1)` *(1-based index: starts at the 2nd element and extracts 1 item → returns `example.com`)*.
   * Preview data to verify the `domain` column displays clean domain strings.

#### Step 3: Add Sink (Silver)
1. Click **+** → Scroll down and select **Sink** → Name: `Silver_Destination`.
2. Click **+ New** next to Dataset:
   * Select **ADLS Gen2** → **Parquet** → Name: `DS_Silver`.
   * **Linked Service:** `LS_DataLake`.
   * **File path:** Container `silver`, Directory `enriched_sales`.
   * **Import schema:** `None`.

#### Step 4: Run & Verify
1. Switch back to the `Transformation_Layer` pipeline tab.
2. Click **Debug** to execute the Data Flow.
3. Verify in ADLS Gen2 that the transformed Parquet file(s) are written to `silver/enriched_sales/`.
4. Click **Publish all**.